In [ ]:
from data_classes.datasets import USdatasetOmni
import pickle
from sklearn.cluster import KMeans
from utils.paths import *
from utils.utils import get_sft_transforms
from transformers import AutoImageProcessor, AutoModel
from transformers import CLIPVisionModel


In [ ]:
filepath = '/media/raid0/US_FiLMUNet/loggings/7b4f82c5799f/kmeans_model.pkl'

with open(filepath, 'rb') as f:
    kmeans_model = pickle.load(f)
    print(f"KMeans model loaded from {filepath}")

In [ ]:
kmeans_model

In [ ]:
dataset = USdatasetOmni(
    base_dir=DATA_DIR,
    split="train_cls",
    transforms=get_sft_transforms(train=False),
    out_size=512,
    data_type="segmentation",
    ccl_crop=False,
    keep_aspect_ratio=True,
    self_norm=True, 
    include_testicles=True, 
    self_id=True,
    use_cluster_id = True,
    # kmeans_model=kmeans_model,
    num_clusters=500,
    enc_type = 'dino'
)

In [ ]:
ids = [item['cluster_id'].item() for item in dataset.items]

In [ ]:
import torch

ids = torch.Tensor(ids)
ids.shape

In [ ]:
ids.unique().shape

In [ ]:
dataset.items[0]

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np
# Extract embeddings and cluster labels
embeddings = np.stack([item['self_id'].numpy() for item in dataset.items])
cluster_labels = np.array([item['cluster_id'] for item in dataset.items])

# Method 1: PCA to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.6)
plt.colorbar(scatter, label='Cluster ID')
plt.title(f'K-means Clusters (k={dataset.num_clusters}) - PCA')
plt.xlabel('PC 1')
plt.ylabel('PC 2')

# Plot cluster centers
centers_2d = pca.transform(dataset.kmeans_model.cluster_centers_)
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], 
           c='red', marker='X', s=200, edgecolors='black', label='Centroids')
plt.legend()
plt.show()

# Method 2: t-SNE (better for visualization but slower)
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_tsne = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_tsne[:, 0], embeddings_tsne[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.6)
plt.colorbar(scatter, label='Cluster ID')
plt.title(f'K-means Clusters (k={dataset.num_clusters}) - t-SNE')
plt.show()

In [ ]:
df

In [ ]:
# Count samples per cluster and organ
from collections import defaultdict

cluster_organ_counts = defaultdict(lambda: defaultdict(int))
for item in dataset.items:
    cluster_id = item['cluster_id']
    organ_label = item['organ_label']
    cluster_organ_counts[cluster_id][organ_label] += 1

# Visualize as heatmap
import pandas as pd
import seaborn as sns

df = pd.DataFrame(cluster_organ_counts).fillna(0).T
df = df.sort_index()  # Sort by cluster ID (index)

plt.figure(figsize=(12, 48))
sns.heatmap(df, annot=True, fmt='g', cmap='YlOrRd')
plt.title('Cluster Distribution by Organ Type')
plt.xlabel('Organ')
plt.ylabel('Cluster ID')
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import numpy as np

# Extract embeddings and cluster labels
embeddings = np.stack([item['self_id'].numpy() for item in dataset.items])
cluster_labels = np.array([item['cluster_id'] for item in dataset.items])
organ_labels = [item['organ_label'] for item in dataset.items]

# Apply 3D t-SNE (fixed parameter)
print("Computing 3D t-SNE...")
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=30, max_iter=1000)
embeddings_3d = tsne_3d.fit_transform(embeddings)

# Create interactive 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    mode='markers',
    marker=dict(
        size=4,
        color=cluster_labels,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cluster ID"),
        line=dict(width=0.5, color='white')
    ),
    text=[f'Cluster: {c}<br>Organ: {o}' for c, o in zip(cluster_labels, organ_labels)],
    hovertemplate='<b>%{text}</b><br>x: %{x:.2f}<br>y: %{y:.2f}<br>z: %{z:.2f}<extra></extra>'
)])

fig.update_layout(
    title=f'Interactive 3D t-SNE of K-means Clusters (k={dataset.num_clusters})',
    scene=dict(
        xaxis_title='t-SNE 1',
        yaxis_title='t-SNE 2',
        zaxis_title='t-SNE 3'
    ),
    width=1000,
    height=800
)

fig.show()

In [ ]:
from utils.utils import organ_to_class_dict
# Extract data
embeddings = np.stack([item['self_id'].numpy() for item in dataset.items])
cluster_labels = np.array([item['cluster_id'] for item in dataset.items])
organ_labels = np.array([organ_to_class_dict[item['organ_label']] for item in dataset.items])
organ_names = [item['organ_label'] for item in dataset.items]

# Apply 3D t-SNE
print("Computing 3D t-SNE...")
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=30, max_iter=1000)
embeddings_3d = tsne_3d.fit_transform(embeddings)

# Create interactive 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    mode='markers',
    marker=dict(
        size=4,
        color=organ_labels,  # Changed to organ_labels
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Organ ID"),
        line=dict(width=0.5, color='white')
    ),
    text=[f'Organ: {o}<br>Organ ID: {oid}<br>Cluster: {c}' 
          for o, oid, c in zip(organ_names, organ_labels, cluster_labels)],
    hovertemplate='<b>%{text}</b><br>x: %{x:.2f}<br>y: %{y:.2f}<br>z: %{z:.2f}<extra></extra>'
)])

fig.update_layout(
    title=f'Interactive 3D t-SNE Colored by Organ Type (K-means k={dataset.num_clusters})',
    scene=dict(
        xaxis_title='t-SNE 1',
        yaxis_title='t-SNE 2',
        zaxis_title='t-SNE 3'
    ),
    width=1000,
    height=800
)

fig.show()